# Notebook 06: Hybrid Retrieval and Reranking

**Project:** Advance AI Research Assistant  
**Module:** Information Retrieval and Ranking  
**Notebook:** `06_hybrid_retrieval_and_reranking.ipynb`  
**Status:** In Progress

---

## 1. Overview

This notebook implements the hybrid information retrieval component of the Advance AI Research Assistant.

The system combines semantic search using ChromaDB with keyword-based retrieval using BM25. The results from both retrieval methods are combined through Reciprocal Rank Fusion (RRF) to produce a unified ranking of relevant document passages.

A Cross-Encoder model is subsequently applied to rerank the retrieved passages based on their relevance to the user's query.

This module aims to improve the accuracy and relevance of scientific information retrieval and provides the retrieval foundation for the Retrieval-Augmented Generation (RAG) pipeline.

## 2. Objectives

The main objectives of this notebook are:

- Load the processed scientific document chunks.
- Implement keyword-based retrieval using BM25.
- Connect to the persistent ChromaDB vector database.
- Implement semantic retrieval using text embeddings.
- Combine keyword and semantic search results using Reciprocal Rank Fusion (RRF).
- Apply Cross-Encoder reranking to improve retrieval relevance.
- Test the retrieval pipeline using scientific research questions.
- Preserve source document and page metadata for citation generation.
- Export retrieval evaluation results for further analysis.

## 3. Tools and Technologies

| Component | Technology |
|---|---|
| Programming language | Python 3.11 |
| Keyword retrieval | BM25 (`rank-bm25`) |
| Semantic retrieval | ChromaDB |
| Embedding framework | Sentence Transformers |
| Embedding model | `all-MiniLM-L6-v2` |
| Ranking fusion | Reciprocal Rank Fusion (RRF) |
| Reranking model | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Numerical processing | NumPy |
| Data analysis | Pandas |
| Deep learning framework | PyTorch |

## 4. Input and Expected Output

**Input:**
- `data/processed/document_chunks.jsonl`
- `data/chroma_db/` (persistent ChromaDB database)

**Expected outputs:**
- BM25 keyword retrieval results.
- ChromaDB semantic retrieval results.
- Combined hybrid retrieval rankings.
- Cross-Encoder reranked passages.
- Preserved

## Section 1 – Imports and configuration

In [8]:
# Imports and configuration
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import chromadb
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

In [9]:
# Set up project paths and configuration:
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

SEMANTIC_TOP_K = 10
BM25_TOP_K = 10
RRF_TOP_K = 10
FINAL_TOP_K = 5

RRF_K = 60

print("Project:", PROJECT_ROOT)
print("Chunks file exists:", CHUNKS_FILE.exists())
print("ChromaDB exists:", CHROMA_DIR.exists())
print("CUDA available:", torch.cuda.is_available())

assert CHUNKS_FILE.exists()
assert CHROMA_DIR.exists()

Project: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Chunks file exists: True
ChromaDB exists: True
CUDA available: True


## Section 2 – Load document chunks
We load the same chunks used to create the embeddings in Notebook 05. The chunk IDs must remain identical so that both retrieval methods refer to the same passages.

In [10]:
# Load document chunks
chunks = []

with CHUNKS_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            chunks.append(json.loads(line))

chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

print("Chunks loaded:", len(chunks))
print("Unique chunk IDs:", len(chunks_by_id))

assert len(chunks) == len(chunks_by_id)

Chunks loaded: 156
Unique chunk IDs: 156


## Section 3 – Initialize BM25 keyword retrieval
BM25 ranks documents according to the occurrence and importance of query terms. It is particularly useful for scientific terminology and exact keywords.

In [11]:
# Initialize BM25 keyword retrieval
def tokenize(text: str) -> list[str]:
    """Convert text into lowercase word tokens."""
    return re.findall(r"\b\w+\b", text.lower())


tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created.")
print("Indexed passages:", len(tokenized_corpus))

BM25 index created.
Indexed passages: 156


In [12]:
# Creating a BM25 search function:
def bm25_search(query: str, top_k: int = BM25_TOP_K) -> list:
    """Retrieve passages using BM25 keyword matching."""

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        chunk = chunks[index]

        results.append({
            "chunk_id": chunk["chunk_id"],
            "score": float(scores[index]),
            "text": chunk["text"],
            "metadata": {
                "filename": chunk["filename"],
                "page_number": chunk["page_number"],
                "document_id": chunk["document_id"]
            }
        })

    return results

In [13]:
# Testing BM25
test_query = "How do deep neural networks identify different cell types?"

bm25_results = bm25_search(test_query)

for rank, result in enumerate(bm25_results[:5], start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Filename:", result["metadata"]["filename"])
    print("Page:", result["metadata"]["page_number"])
    print("BM25 score:", round(result["score"], 4))
    print("Text:", result["text"][:300])

Rank: 1
Filename: 41598_2019_Article_38798.pdf
Page: 1
BM25 score: 9.0801
Text: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seye
Rank: 2
Filename: 41598_2019_Article_38798.pdf
Page: 1
BM25 score: 7.9274
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to technical replicates of the same experiment. Although we use an unsupervised approa
Rank: 3
Filename: 41598_2019_Article_38798.pdf
Page: 1
BM25 score: 7.335
Text: uripotent stem cells (iPSCs), to some specific tissue required for a patient, such as neurons or cardiomyocytes1. After treating the stem cells with a differen

## Section 4 – Connect to ChromaDB and load the embedding model
We reuse the existing persistent ChromaDB database and embedding model from Notebook 05. We do not create another collection or regenerate document embeddings.

In [14]:
# Connect to ChromaDB and load the embedding model
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

collection = chroma_client.get_collection(
    name="research_papers"
)

print("Collection:", collection.name)
print("Stored vectors:", collection.count())

assert collection.count() == len(chunks)

Collection: research_papers
Stored vectors: 156


In [15]:
# Load the embedding model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded.")
print("Device:", DEVICE)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Device: cuda


## Section 5 – Implement semantic retrieval
This function converts a user query into an embedding and searches ChromaDB for the most similar passages.

In [16]:
def semantic_search(
    query: str,
    top_k: int = SEMANTIC_TOP_K
) -> list:
    """Retrieve passages using ChromaDB semantic similarity."""

    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for i, chunk_id in enumerate(results["ids"][0]):
        retrieved.append({
            "chunk_id": chunk_id,
            "score": float(results["distances"][0][i]),
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i]
        })

    return retrieved

In [17]:
# Test semantic retrieval
semantic_results = semantic_search(test_query)

for rank, result in enumerate(semantic_results[:5], start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Filename:", result["metadata"]["filename"])
    print("Page:", result["metadata"]["page_number"])
    print("Cosine distance:", round(result["score"], 4))
    print("Text:", result["text"][:300])

Rank: 1
Filename: 41598_2019_Article_38798.pdf
Page: 5
Cosine distance: 0.4013
Text: cell type, while other CICs of the same sample could not be easily classified. In order to guide the training process of autoencoder to identify those neural network parameters that could produce more easily classifiable cell identity codes, we incorporated some additional layers to the network arch
Rank: 2
Filename: 41598_2019_Article_38798.pdf
Page: 1
Cosine distance: 0.4038
Text: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seye
Rank: 3
Filename: 41598_2019_Article_38798.pdf
Page: 1
Cosine distance: 0.419
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The or

## Section 6 – Combine results using Reciprocal Rank Fusion (RRF)
BM25 and semantic retrieval use different scoring systems. Their raw scores should not simply be added together. Instead, we use Reciprocal Rank Fusion. RRF combines the rankings rather than comparing the original scores.

The formula is:
\[
RRF(d) = \sum_{r \in R} \frac{1}{k + rank_r(d)}
\]

Where:
- \(d\) = retrieved document chunk.
- \(R\) = retrieval methods.
- \(rank_r(d)\) = position of the chunk in a retrieval ranking.
- \(k\) = a constant, typically 60.

In [18]:
# Implement RRF
def reciprocal_rank_fusion(
    result_lists: list,
    k: int = RRF_K,
    top_k: int = RRF_TOP_K
) -> list:
    """Combine multiple retrieval rankings using RRF."""

    rrf_scores = defaultdict(float)
    results_by_id = {}

    for results in result_lists:
        for rank, result in enumerate(results, start=1):

            chunk_id = result["chunk_id"]

            rrf_scores[chunk_id] += 1.0 / (k + rank)

            results_by_id[chunk_id] = result

    ranked_ids = sorted(
        rrf_scores,
        key=lambda chunk_id: rrf_scores[chunk_id],
        reverse=True
    )

    fused_results = []

    for chunk_id in ranked_ids[:top_k]:
        result = results_by_id[chunk_id].copy()

        result["rrf_score"] = rrf_scores[chunk_id]

        fused_results.append(result)

    return fused_results

In [19]:
# Combine BM25 and semantic results
hybrid_results = reciprocal_rank_fusion(
    [bm25_results, semantic_results]
)

print("Hybrid retrieval completed.")
print("Retrieved passages:", len(hybrid_results))

for rank, result in enumerate(hybrid_results, start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Chunk ID:", result["chunk_id"])
    print("Filename:", result["metadata"]["filename"])
    print("Page:", result["metadata"]["page_number"])
    print("RRF score:", round(result["rrf_score"], 6))
    print("Text:", result["text"][:300])

Hybrid retrieval completed.
Retrieved passages: 10
Rank: 1
Chunk ID: 41598_2019_Article_38798_p1_c1
Filename: 41598_2019_Article_38798.pdf
Page: 1
RRF score: 0.032522
Text: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seye
Rank: 2
Chunk ID: 41598_2019_Article_38798_p1_c2
Filename: 41598_2019_Article_38798.pdf
Page: 1
RRF score: 0.032002
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to technical replicates of the same experiment. Although we use an unsupervised approa
Rank: 3
Chunk ID: 41598_2019_Article_38798_p5_c6
Filename: 41598_2019_Article_38798.pdf
Page: 5
RRF sco

## Section 7 – Cross-Encoder reranking
RRF combines the two retrieval rankings. However, we can further improve the ordering using a Cross-Encoder. Unlike the embedding model, which encodes the query and passage separately, a Cross-Encoder evaluates the query and passage together. It produces a relevance score for each pair.

For this project, we use:
cross-encoder/ms-marco-MiniLM-L-6-v2

We run the reranker on the CPU to avoid unnecessary GPU memory pressure.

In [20]:
reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    device="cpu",
    max_length=512
)

print("Cross-Encoder loaded successfully.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder loaded successfully.


In [21]:
# Implement the reranking function
def rerank_results(
    query: str,
    results: list,
    top_k: int = FINAL_TOP_K
) -> list:
    """Rerank retrieved passages using a Cross-Encoder."""

    if not results:
        return []

    pairs = [
        (query, result["text"])
        for result in results
    ]

    scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )

    reranked = []

    for result, score in zip(results, scores):
        item = result.copy()
        item["rerank_score"] = float(score)
        reranked.append(item)

    reranked.sort(
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [22]:
# Test reranking
reranked_results = rerank_results(
    test_query,
    hybrid_results
)

for rank, result in enumerate(reranked_results, start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Filename:", result["metadata"]["filename"])
    print("Page:", result["metadata"]["page_number"])
    print("RRF score:", round(result["rrf_score"], 6))
    print("Cross-Encoder score:", round(result["rerank_score"], 4))
    print("Text:", result["text"][:400])

Rank: 1
Filename: 41598_2019_Article_38798.pdf
Page: 11
RRF score: 0.015385
Cross-Encoder score: 3.2647
Text: www.nature.com/scientificreports/ Figure 8. Comparison of 10 different architectures of the deep neural networks in encoding the expression levels of 1000 genes into the cell identity code (CIC) of size 30 and decoding it back to the original values. The numbers following letter L determine the number of neurons after a linear layer, e.g. “L300, G, L30” is a linear layer that fully connects the in
Rank: 2
Filename: 41598_2019_Article_38798.pdf
Page: 1
RRF score: 0.032002
Cross-Encoder score: 3.1581
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to technical replicates of the same experiment. Although we use an unsupervised approach to train the autoencoder, we show different values of the CIC are connected t

**Important:** Cross-Encoder scores are relevance scores, not probabilities. They should be used to rank passages, not interpreted as percentages.

## Section 8 – Create the complete hybrid retrieval pipeline
Now we combine all three stages into one reusable function:
1. BM25 retrieval.
2. Semantic retrieval.
3. RRF fusion.
4. Cross-Encoder reranking.
This is the function that our backend will eventually call when a user submits a question.

In [23]:
def hybrid_retrieve(
    query: str,
    final_top_k: int = FINAL_TOP_K
) -> list:
    """Complete hybrid retrieval and reranking pipeline."""

    # Step 1: Keyword retrieval
    keyword_results = bm25_search(
        query,
        top_k=BM25_TOP_K
    )

    # Step 2: Semantic retrieval
    vector_results = semantic_search(
        query,
        top_k=SEMANTIC_TOP_K
    )

    # Step 3: Combine rankings using RRF
    fused_results = reciprocal_rank_fusion(
        [keyword_results, vector_results],
        top_k=RRF_TOP_K
    )

    # Step 4: Rerank using Cross-Encoder
    final_results = rerank_results(
        query,
        fused_results,
        top_k=final_top_k
    )

    return final_results

In [25]:
# Test the complete pipeline
final_results = hybrid_retrieve(test_query)

print("Query:", test_query)
print("Final retrieved passages:", len(final_results))

for rank, result in enumerate(final_results, start=1):
    print("=" * 80)
    print("Final Rank:", rank)
    print("Filename:", result["metadata"]["filename"])
    print("Page:", result["metadata"]["page_number"])
    print("Reranking score:", round(result["rerank_score"], 4))
    print("Text:", result["text"][:500])

Query: How do deep neural networks identify different cell types?
Final retrieved passages: 5
Final Rank: 1
Filename: 41598_2019_Article_38798.pdf
Page: 11
Reranking score: 3.2647
Text: www.nature.com/scientificreports/ Figure 8. Comparison of 10 different architectures of the deep neural networks in encoding the expression levels of 1000 genes into the cell identity code (CIC) of size 30 and decoding it back to the original values. The numbers following letter L determine the number of neurons after a linear layer, e.g. “L300, G, L30” is a linear layer that fully connects the input 1000 neurons to 300 neurons, followed by a layer that applies a LogSigmoid function to each of t
Final Rank: 2
Filename: 41598_2019_Article_38798.pdf
Page: 1
Reranking score: 3.1581
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to techn

## Section 9 – Evaluate using multiple research questions
We now test the retrieval pipeline with three questions related to the scientific papers in our collection.

In [26]:
test_queries = [
    "How do deep neural networks help identify different cell types?",
    "What is the Transformer architecture and how does self-attention work?",
    "How can machine learning be used for cancer classification?"
]

evaluation_results = []

for query in test_queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)

    results = hybrid_retrieve(query)

    for rank, result in enumerate(results, start=1):

        metadata = result["metadata"]

        evaluation_results.append({
            "query": query,
            "rank": rank,
            "chunk_id": result["chunk_id"],
            "filename": metadata["filename"],
            "page_number": metadata["page_number"],
            "rrf_score": result["rrf_score"],
            "rerank_score": result["rerank_score"],
            "text_preview": result["text"][:300]
        })

        print(
            f"Rank {rank}: "
            f"{metadata['filename']} | "
            f"Page {metadata['page_number']} | "
            f"Score {result['rerank_score']:.4f}"
        )


QUERY: How do deep neural networks help identify different cell types?
Rank 1: 41598_2019_Article_38798.pdf | Page 1 | Score 3.3362
Rank 2: 41598_2019_Article_38798.pdf | Page 1 | Score 2.4999
Rank 3: 41598_2019_Article_38798.pdf | Page 2 | Score 0.5565
Rank 4: 41598_2019_Article_38798.pdf | Page 5 | Score -0.2652
Rank 5: s41598-019-52937-5.pdf | Page 2 | Score -0.4173

QUERY: What is the Transformer architecture and how does self-attention work?
Rank 1: NIPS-2017-attention-is-all-you-need-Paper.pdf | Page 2 | Score 5.0069
Rank 2: NIPS-2017-attention-is-all-you-need-Paper.pdf | Page 2 | Score 4.5080
Rank 3: NIPS-2017-attention-is-all-you-need-Paper.pdf | Page 2 | Score 2.6579
Rank 4: NIPS-2017-attention-is-all-you-need-Paper.pdf | Page 9 | Score 2.6309
Rank 5: NIPS-2017-attention-is-all-you-need-Paper.pdf | Page 3 | Score 1.7844

QUERY: How can machine learning be used for cancer classification?
Rank 1: s41598-019-52937-5.pdf | Page 2 | Score 5.1064
Rank 2: s41598-019-52937-5.pdf | Pa

## Section 10 – Save retrieval results

In [27]:
evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

RESULTS_FILE = OUTPUT_DIR / "retrieval_test_results.csv"

evaluation_df.to_csv(
    RESULTS_FILE,
    index=False,
    encoding="utf-8"
)

print("Retrieval results saved:")
print(RESULTS_FILE)

print("Total queries:", evaluation_df["query"].nunique())
print("Total retrieved passages:", len(evaluation_df))

,query,rank,chunk_id,filename,page_number,rrf_score,rerank_score,text_preview
0,How do deep neural networks help identify diff...,1,41598_2019_Article_38798_p1_c1,41598_2019_Article_38798.pdf,1,0.032787,3.336152,www.nature.com/scientificreports OPEN Cell Ide...
1,How do deep neural networks help identify diff...,2,41598_2019_Article_38798_p1_c2,41598_2019_Article_38798.pdf,1,0.032258,2.499890,ific structure of deep autoencoders that can e...
2,How do deep neural networks help identify diff...,3,41598_2019_Article_38798_p2_c1,41598_2019_Article_38798.pdf,2,0.029911,0.556461,www.nature.com/scientificreports/ Kullback-Lei...
3,How do deep neural networks help identify diff...,4,41598_2019_Article_38798_p5_c6,41598_2019_Article_38798.pdf,5,0.031025,-0.265186,"cell type, while other CICs of the same sample..."
4,How do deep neural networks help identify diff...,5,s41598-019-52937-5_p2_c4,s41598-019-52937-5.pdf,2,0.029877,-0.417309,"NN) based classifier, uses somatic point mutat..."
5,What is the Transformer architecture and how d...,1,NIPS-2017-attention-is-all-you-need-Paper_p2_c3,NIPS-2017-attention-is-all-you-need-Paper.pdf,2,0.031258,5.006880,this is reduced to a constant number of operat...
6,What is the Transformer architecture and how d...,2,NIPS-2017-attention-is-all-you-need-Paper_p2_c4,NIPS-2017-attention-is-all-you-need-Paper.pdf,2,0.032787,4.507971,"Ns or convolution. In the following sections, ..."
7,What is the Transformer architecture and how d...,3,NIPS-2017-attention-is-all-you-need-Paper_p2_c2,NIPS-2017-attention-is-all-you-need-Paper.pdf,2,0.015152,2.657939,nisms are used in conjunction with a recurrent...
8,What is the Transformer architecture and how d...,4,NIPS-2017-attention-is-all-you-need-Paper_p9_c2,NIPS-2017-attention-is-all-you-need-Paper.pdf,9,0.030118,2.630927,bility function than dot product may be benefi...
9,What is the Transformer architecture and how d...,5,NIPS-2017-attention-is-all-you-need-Paper_p3_c1,NIPS-2017-attention-is-all-you-need-Paper.pdf,3,0.030415,1.784447,Figure 1: The Transformer - model architecture...


Retrieval results saved:
/home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/processed/retrieval_test_results.csv
Total queries: 3
Total retrieved passages: 15


## 5. Conclusion

The hybrid retrieval pipeline was implemented by combining BM25 keyword retrieval and ChromaDB semantic retrieval.

Reciprocal Rank Fusion (RRF) was used to combine the rankings from both retrieval methods. A Cross-Encoder was then applied to rerank the retrieved passages according to their relevance to the original query.

The pipeline was tested using scientific research questions covering cell identity, Transformer architecture, and cancer classification.

**Key outcomes:**
- BM25 keyword retrieval implemented.
- Semantic retrieval integrated with ChromaDB.
- Reciprocal Rank Fusion implemented.
- Cross-Encoder reranking integrated.
- Source document and page metadata preserved.
- Multiple research queries tested.
- Retrieval results exported to CSV.

This module completes the retrieval component of the Advance AI Research Assistant and prepares the project for integration with a Large Language Model (LLM) in the next development stage.

## 6. Retrieval Evaluation

The hybrid retrieval pipeline was tested using three scientific research questions covering cell identity, Transformer architecture, and cancer classification.

The evaluation demonstrated that the system successfully retrieved relevant passages from the corresponding scientific papers.

**Evaluation results:**

| Research topic | Retrieved source | Top-ranked result |
|---|---|---|
| Cell identity and deep neural networks | `41598_2019_Article_38798.pdf` | Relevant |
| Transformer architecture and self-attention | `NIPS-2017-attention-is-all-you-need-Paper.pdf` | Relevant |
| Machine learning for cancer classification | `s41598-019-52937-5.pdf` | Relevant |

The results demonstrate that combining BM25 keyword retrieval with ChromaDB semantic search and Reciprocal Rank Fusion provides a functional retrieval pipeline. Cross-Encoder reranking further improves the ordering of the retrieved passages.

This evaluation is qualitative and based on a small set of manually selected test questions. A formal retrieval benchmark with annotated relevance judgments would be required for quantitative performance evaluation.

### Limitations

- The evaluation uses only three test questions.
- Multiple retrieved passages can originate from the same document.
- Character-based chunking can occasionally split words between chunks.
- Cross-Encoder scores represent ranking signals rather than probabilities.

Despite these limitations, the retrieval pipeline meets the initial MVP requirements and is ready for integration into the RAG application.